In [2]:
import pandas as pd
import serpapi
import time

# --- CONFIGURATION ---
# Your API key from Phase 1
API_KEY = "35ee400445b907086d64bf97daa4d3a981019d97a68f3a9f661bda1ffeb7daa5"

def get_image_from_title(title):
    """Searches Google Images for the article headline and returns the top image."""
    client = serpapi.Client(api_key=API_KEY)
    try:
        results = client.search({
            "engine": "google",
            "q": title,
            "tbm": "isch",  # Forces Google Image Search
            "gl": "in",
            "hl": "en",
            "num": 1        # We only need the #1 result
        })
        
        images_results = results.get("images_results", [])
        if images_results:
            # 'original' provides the high-resolution direct image link
            return images_results[0].get("original") 
            
        return "No image found"
    except Exception as e:
        print(f"  [!] SerpApi error: {e}")
        return "Error"

def process_images_via_search(input_filename, output_filename):
    print(f"Loading data from {input_filename}...")
    
    try:
        df = pd.read_csv(input_filename)
    except FileNotFoundError:
        print(f"Error: Could not find {input_filename}")
        return

    print(f"Searching Google Images for {len(df)} headlines...\n")
    
    image_urls = []
    
    for index, row in df.iterrows():
        title = row['Title']
        print(f"[{index + 1}/{len(df)}] Searching for: {title[:50]}...")
        
        img_url = get_image_from_title(title)
        image_urls.append(img_url)
        
        # Brief pause to respect API rate limits
        time.sleep(1)

    # Overwrite the broken Image URL column if it exists, or create a new one
    df['Image URL'] = image_urls
    
    # Reorder columns to put Image URL next to the summary
    cols = df.columns.tolist()
    if 'Magazine Summary' in cols:
        cols.insert(cols.index('Magazine Summary') + 1, cols.pop(cols.index('Image URL')))

    # Save the updated dataset
    df.to_csv(output_filename, index=False, encoding='utf-8')
    
    success_count = len([url for url in image_urls if url.startswith('http')])
    print(f"\nSUCCESS: Found unique images for {success_count} out of {len(df)} articles.")
    print(f"Saved dataset to: {output_filename}")

if __name__ == "__main__":
    INPUT_FILE = "Magazine_45_Pages_SPLIT_COLUMNS.csv"
    OUTPUT_FILE = "Magazine_50_Pages_Final_Images.csv"
    
    process_images_via_search(INPUT_FILE, OUTPUT_FILE)

Loading data from Magazine_45_Pages_SPLIT_COLUMNS.csv...
Searching Google Images for 45 headlines...

[1/45] Searching for: Army, Assam Police review security framework for I...
[2/45] Searching for: Army Chief General Dhiraj Seth Reviews Security Si...
[3/45] Searching for: Army Chief Reviews Peace Initiatives in Manipur...
[4/45] Searching for: Multilateral joint military exercise in Meghalaya ...
[5/45] Searching for: Fake Indian Army Vehicle Busted in Manipur With 15...
[6/45] Searching for: Lieutenant Colonel Maneo Francis PF of 21 Para (SF...
[7/45] Searching for: Indian Army chief General Dhiraj Seth reviews batt...
[8/45] Searching for: Indian Army swiftly neutralises UXOs following sta...
[9/45] Searching for: Dogs of War: K9 units of Indian Army conduct helib...
[10/45] Searching for: Army troops undertake route march at 17,000 feet i...
[11/45] Searching for: After ambushes, CRPF tells forces in Manipur to us...
[12/45] Searching for: After Trishul, why is India holding Poor